# Save and publish a model bundle

Training produces a fitted pipeline in `result["model"]`. This notebook
turns it into a self-describing bundle and shows how to publish it:

1. train a small model on synthetic recordings (the same pipeline as
   `01_training.ipynb`);
2. write `model.joblib`, `metadata.json` and `model_card.md` with
   `save_model`;
3. upload the bundle to the Hugging Face Hub, guarded by `HF_TOKEN` so the
   notebook runs offline when no token is configured.

The bundle stores the exact input contract (electrodes, window length,
sample rates), the encoder identity and the preprocessing snapshot. REVE
weights are **not** uploaded; inference re-downloads them by id.


In [1]:
import json
import os
import tempfile
from pathlib import Path

from IPython.display import Markdown, display

from hotb_2026.classification.dataset import (
    build_dataset,
    make_synthetic_recording,
)
from hotb_2026.classification.training import TrainingConfig, train
from hotb_2026.features import ReveEncoder
from hotb_2026.model_store import (
    StoreConfig,
    publish,
    read_metadata,
    save_model,
)

/Users/dominikmika/PycharmProjects/HOTB_2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Train a small model

Six synthetic sessions are enough for a demo. With five or more sessions
`group_by="auto"` holds out whole sessions, so the reported score is not
inflated by overlapping windows.


In [2]:
workdir = Path(tempfile.mkdtemp(prefix="hotb-model-store-"))
recordings = [
    make_synthetic_recording(
        duration_s=120.0,
        state_duration_s=30.0,
        seed=index,
        session_id=f"demo-{index + 1}",
    )
    for index in range(6)
]
dataset = build_dataset(recordings, window_s=10.0, group_by="auto")

result = train(
    dataset,
    config=TrainingConfig(n_splits=3, n_trials=5, seed=0),
    encoder=ReveEncoder(dataset.electrodes),
    cache_dir=str(workdir / "embeddings"),
)

print("windows     :", dataset.n_windows)
print("CV mean     :", round(result["cv"]["mean"], 3))
print("holdout BA  :", round(result["holdout"]["metrics"]["balanced_accuracy"], 3))

W1115 07:27:51.120000 28093 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 1/1 [00:00<00:00, 4860.14it/s]

flash_attn not found, install it with `pip install flash_attn` if you want to use it


Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 140/140 [00:00<00:00, 20295.25it/s]

windows     : 72
CV mean     : 0.685
holdout BA  : 0.667


## 2. Save the bundle

`save_model` validates the pipeline before writing anything: it must be
fitted, its classes must be `0..n-1`, and its feature count must match the
embedding dimension recorded during training.

The version (`0.1.0` here) is written to `metadata.json` and later used as
the Hugging Face git tag.


In [3]:
bundle = save_model(
    result,
    workdir / "bundle",
    config=StoreConfig(version="0.1.0"),
)

print("bundle:", bundle.directory)
for path in bundle.files:
    print(f"  {path.name:16} {path.stat().st_size / 1024:8.1f} KiB")

bundle: /var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-model-store-e4s5wz9q/bundle
  metadata.json         3.8 KiB
  model.joblib        157.7 KiB
  model_card.md         1.6 KiB


`metadata.json` is the contract inference reads. It records the electrode
order, window length, sample rates, encoder id and dimensions, the exact
preprocessing snapshot, and the training metrics.


In [4]:
metadata = read_metadata(bundle.directory)
contract = {
    key: metadata[key]
    for key in (
        "model_version",
        "labels",
        "input",
        "encoder",
        "preprocessing",
    )
}
display(Markdown("```json\n" + json.dumps(contract, indent=2) + "\n```"))

```json
{
  "model_version": "0.1.0",
  "labels": {
    "classes": [
      0,
      1,
      2
    ],
    "n_classes": 3,
    "names": [
      "low",
      "medium",
      "high"
    ]
  },
  "input": {
    "electrodes": [
      "F3",
      "F4",
      "C3",
      "C4",
      "P3",
      "P4",
      "O1",
      "O2"
    ],
    "label_rule": "state active at window start",
    "n_samples": 2000,
    "raw_sfreq": 250.0,
    "sfreq": 200.0,
    "time_units": "s",
    "window_s": 10.0
  },
  "encoder": {
    "embedding_dim": 512,
    "model_name": "brain-bzh/reve-base",
    "patch_size": 200,
    "pos_model_name": "brain-bzh/reve-positions",
    "revision": null
  },
  "preprocessing": {
    "bandpass": [
      0.5,
      45.0
    ],
    "clip": 15.0,
    "filter_order": 4,
    "notch_freq": 50.0,
    "notch_q": 30.0,
    "sfreq": 250.0,
    "target_sfreq": 200.0,
    "zscore": true
  }
}
```

In [5]:
display(Markdown(bundle.model_card_path.read_text()))

---
library_name: scikit-learn
tags:
  - eeg
  - neuroscience
  - cognitive-load
  - reve
  - scikit-learn
---

# Cognitive-overload classifier (REVE + SVM)

3-class cognitive-overload model for N-back EEG. A frozen REVE-Base transformer embeds each window; a calibrated RBF SVM classifies the embedding.

| | |
|---|---|
| Version | `0.1.0` |
| Classes | low, medium, high |
| Encoder | `brain-bzh/reve-base` |
| Position encoder | `brain-bzh/reve-positions` |
| Embedding dim | 512 |
| Patch size | 200 |
| Window | 10 s at 200 Hz |
| Electrodes | F3, F4, C3, C4, P3, P4, O1, O2 |

## Held-out performance

| metric | value |
|---|---|
| Accuracy | 0.75 |
| Balanced accuracy | 0.667 |
| Macro F1 | 0.657 |
| ROC AUC (OVR) | 0.901 |

## Input contract

- Electrodes, in channel order: F3, F4, C3, C4, P3, P4, O1, O2
- Window length: 10 s (2000 samples at 200 Hz)
- Raw sample rate: 250 Hz
- Time units: s
- Label rule: state active at window start

## Preprocessing

| parameter | value |
|---|---|
| `sfreq` | 250 |
| `notch_freq` | 50 |
| `notch_q` | 30 |
| `bandpass` | [0.5, 45.0] |
| `filter_order` | 4 |
| `target_sfreq` | 200 |
| `zscore` | True |
| `clip` | 15 |

## Usage

```python
import joblib

pipeline = joblib.load("model.joblib")
probabilities = pipeline.predict_proba(embedding)
label = pipeline.predict(embedding)[0]
```

`embedding` must come from the same frozen encoder and window preprocessing recorded in `metadata.json`.

## Files

- `model.joblib` — fitted `Pipeline(StandardScaler, CalibratedClassifierCV(SVC))`
- `metadata.json` — input contract, encoder, preprocessing and metrics
- `model_card.md` — this file



## 3. Publish to Hugging Face

Set `HF_TOKEN` (a write token) and `HOTB_MODEL_REPO_ID`
(`owner/name`, for example `ai-med/hotb-2026-nback-classifier`) to upload.
The target is a **public** model repo by default; if the configured owner
is not writable, `publish` falls back to the authenticated user's
namespace.

The cell below is a no-op when those variables are unset, so the notebook
runs without network access or credentials.


In [6]:
repo_id = os.environ.get("HOTB_MODEL_REPO_ID")
token = os.environ.get("HF_TOKEN")

if token and repo_id:
    summary = publish(bundle.directory, StoreConfig(repo_id=repo_id))
    print(json.dumps(summary, indent=2))
else:
    print(
        "Skipped upload: set HF_TOKEN and HOTB_MODEL_REPO_ID to publish "
        f"the bundle saved at {bundle.directory}."
    )

Skipped upload: set HF_TOKEN and HOTB_MODEL_REPO_ID to publish the bundle saved at /var/folders/8d/gk38hstj1gg1tk319gmlzgm00000gn/T/hotb-model-store-e4s5wz9q/bundle.


## CLI

Saving needs the in-memory training result, so it stays a Python call.
Inspecting and publishing an existing bundle are available from the CLI:

```bash
# print metadata.json
uv run python -m hotb_2026.model_store show bundles/nback

# upload a saved bundle
HF_TOKEN=... uv run python -m hotb_2026.model_store publish bundles/nback \
    --repo-id ai-med/hotb-2026-nback-classifier
```

## Notes

- `model.joblib` is a compressed `Pipeline(StandardScaler,
  CalibratedClassifierCV(SVC))`. Load it with `joblib.load`.
- The bundle version is authoritative: `publish` uses the value from
  `metadata.json` for the tag and commit message.
- Only relative, portable information is stored. No absolute paths,
  tokens or REVE weights end up in the bundle.
- Live inference will rebuild the encoder from `metadata.json` and run
  the recorded preprocessing before calling the pipeline.
